# LightGBM Brain
Output: `simulation_brain_lgbm.pkl`

Comparision:
- C4.5 (mss=10): Overall=0.4886, Balanced=0.2831
- RF (balanced): Overall=0.2880, Balanced=0.3647

In [15]:
import sys
import os
import pm4py
import pandas as pd
import numpy as np
import pickle
from pm4py.algo.conformance.tokenreplay import algorithm as token_replay
from pm4py.algo.discovery.inductive import algorithm as inductive_miner
from pm4py.objects.conversion.process_tree import converter as pt_converter
from pm4py.objects.petri_net.semantics import enabled_transitions, execute
from lightgbm import LGBMClassifier
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report

In [16]:
current_dir = os.getcwd()
sim_core_path = os.path.abspath(os.path.join(current_dir, '..', 'sim_core'))
if sim_core_path not in sys.path:
    sys.path.append(sim_core_path)
bpmn_path = "../data/process_model.bpmn"
log_path = "../data/BPI Challenge 2017.xes.gz"

In [17]:
log = pm4py.read_xes(log_path)

parsing log, completed traces :: 100%|██████████| 31509/31509 [00:51<00:00, 616.59it/s] 


In [18]:
df_log = pm4py.convert_to_dataframe(log)
df_log['time:timestamp'] = pd.to_datetime(df_log['time:timestamp'])
df_log = df_log.sort_values(['case:concept:name', 'time:timestamp'])

case_attr_cols = ["case:concept:name", "case:LoanGoal", "case:ApplicationType", "case:RequestedAmount"]
available_cols = [c for c in case_attr_cols if c in df_log.columns]
case_attrs = df_log.groupby("case:concept:name").first()[available_cols[1:]].reset_index()
case_attrs.columns = ["case_id", "loan_goal", "application_type", "requested_amount"]
case_attrs["loan_goal"] = case_attrs["loan_goal"].fillna("Unknown").astype(str)
case_attrs["application_type"] = case_attrs["application_type"].fillna("Unknown").astype(str)
case_attrs["requested_amount"] = pd.to_numeric(case_attrs["requested_amount"], errors="coerce").fillna(0)
case_attrs["amount_category"] = pd.cut(
    case_attrs["requested_amount"],
    bins=[0, 5000, 10000, 20000, 50000, float("inf")],
    labels=["very_low", "low", "medium", "high", "very_high"]
).astype(str)
case_lookup = case_attrs.set_index("case_id").to_dict("index")

offer_events = df_log[df_log["concept:name"] == "O_Create Offer"].copy()
credit_cols = [c for c in ["CreditScore", "OfferedAmount"] if c in offer_events.columns]
if credit_cols:
    case_credit = offer_events.groupby("case:concept:name")[credit_cols].max().reset_index()
    case_credit.columns = ["case_id"] + credit_cols
    for _, row in case_credit.iterrows():
        cid = row["case_id"]
        if cid in case_lookup and "CreditScore" in credit_cols:
            cs = row.get("CreditScore", None)
            if pd.notna(cs):
                cs = float(cs)
                if cs > 800:   case_lookup[cid]["credit_score_bin"] = "excellent"
                elif cs > 600: case_lookup[cid]["credit_score_bin"] = "good"
                elif cs > 400: case_lookup[cid]["credit_score_bin"] = "fair"
                else:          case_lookup[cid]["credit_score_bin"] = "poor"
print(f"Case attributes loaded for {len(case_lookup)} cases.")

Case attributes loaded for 31509 cases.


In [19]:
tree = inductive_miner.apply(log)
net, initial_marking, final_marking = pt_converter.apply(tree)
replayed_log = token_replay.apply(log, net, initial_marking, final_marking)
event_log = pm4py.convert_to_event_log(log)

replaying log with TBR, completed traces :: 100%|██████████| 15930/15930 [01:13<00:00, 215.87it/s]


In [20]:
decision_training_data = []

for trace_idx, trace in enumerate(event_log):
    replay_result = replayed_log[trace_idx]
    fired_transitions = replay_result['activated_transitions']
    event_idx = 0
    case_id = trace.attributes.get('concept:name')
    case_start_time = trace[0]['time:timestamp']
    current_marking = initial_marking.copy()
    c_info = case_lookup.get(case_id, {})
    loan_goal = c_info.get("loan_goal", "Unknown")
    app_type = c_info.get("application_type", "Unknown")
    amount_cat = c_info.get("amount_category", "medium")
    credit_bin = c_info.get("credit_score_bin", "unknown")
    activity_counts = {}
    offer_count = 0
    rejection_count = 0
    accepted_offer = False

    for t in fired_transitions:
        enabled = enabled_transitions(net, current_marking)
        if len(enabled) > 1:
            if event_idx > 0 and event_idx < len(trace):
                prev_event = trace[event_idx - 1]
                curr_event = trace[event_idx]
                decision_label = t.label if t.label else t.name
                prev_act = prev_event['concept:name']
                features_row = {
                    "case_id": case_id,
                    "prev_activity": prev_act,
                    "last_resource": prev_event.get('org:resource', 'System'),
                    "last_duration": (curr_event['time:timestamp'] - prev_event['time:timestamp']).total_seconds(),
                    "case_duration_hours": (curr_event['time:timestamp'] - case_start_time).total_seconds() / 3600,
                    "target_decision": decision_label,
                    "loan_goal": loan_goal,
                    "application_type": app_type,
                    "amount_category": amount_cat,
                    "credit_score_bin": credit_bin,
                    "offer_count": offer_count,
                    "has_rejection": 1 if rejection_count > 0 else 0,
                    "has_accepted_offer": 1 if accepted_offer else 0,
                    "is_repeated": str(activity_counts.get(prev_act, 0) > 1),
                }
                decision_training_data.append(features_row)
        current_marking = execute(t, net, current_marking)
        if t.label is not None:
            act_name = t.label
            activity_counts[act_name] = activity_counts.get(act_name, 0) + 1
            if act_name in ("O_Created", "O_Create Offer"): offer_count += 1
            if act_name in ("O_Refused", "O_Cancelled", "A_Denied", "A_Cancelled"): rejection_count += 1
            if act_name == "O_Accepted": accepted_offer = True
            event_idx += 1

df_xor_decisions = pd.DataFrame(decision_training_data)
print(f"Toplam Karar Satiri: {len(df_xor_decisions)}")

Toplam Karar Satiri: 2819057


In [21]:
df_clean = df_xor_decisions[
    (df_xor_decisions['target_decision'].notna()) &
    (df_xor_decisions['target_decision'] != "") &
    (~df_xor_decisions['target_decision'].str.contains('tau|skip|init|loop', case=False))
].copy()
counts = df_clean['target_decision'].value_counts()
significant_targets = counts[counts > 50].index
df_clean = df_clean[df_clean['target_decision'].isin(significant_targets)]
print(f"After filtering: {len(df_clean):,} rows, {df_clean['target_decision'].nunique()} targets")

After filtering: 891,414 rows, 24 targets


In [22]:
df_clean['duration_bin'], duration_bins = pd.qcut(
    df_clean['last_duration'], q=5,
    labels=['VeryShort', 'Short', 'Medium', 'Long', 'VeryLong'],
    duplicates='drop', retbins=True
)
df_clean['case_age_category'], case_age_bins = pd.qcut(
    df_clean['case_duration_hours'], q=4,
    labels=['Very_New', 'New', 'Old', 'Delayed'],
    duplicates='drop', retbins=True
)
df_clean['offer_category'] = df_clean['offer_count'].apply(
    lambda x: 'none' if x == 0 else ('single' if x == 1 else 'multiple')
)
print("Binning done.")

Binning done.


In [23]:
features = [
    "prev_activity",
    "duration_bin",
    "case_age_category",
    "loan_goal",
    "application_type",
    "amount_category",
    "credit_score_bin",
    "offer_category",
    "has_rejection",
    "has_accepted_offer",
    "is_repeated",
]

X = df_clean[features].astype(str)
y = df_clean["target_decision"]

In [24]:
MIN_SUPPORT = 100
class_counts = y.value_counts()
valid_classes = class_counts[class_counts >= MIN_SUPPORT].index
mask = y.isin(valid_classes)
X_filtered = X[mask]
y_filtered = y[mask]

removed = sorted(class_counts[class_counts < MIN_SUPPORT].index.tolist())
print(f"Removed {len(removed)} low-support classes: {removed}")
print(f"Kept {y_filtered.nunique()} classes, {len(y_filtered):,} rows")

X_train, X_test, y_train, y_test = train_test_split(
    X_filtered, y_filtered, test_size=0.2, random_state=42, stratify=y_filtered
)

# OrdinalEncoder: .values ensures numpy arrays → no feature name warning
encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_train_enc = encoder.fit_transform(X_train.values)
X_test_enc = encoder.transform(X_test.values)

# LightGBM without class_weight — gradient boosting is already stronger than C4.5
# class_weight='balanced' destroyed W_ classes (same as RF) → removed
lgbm = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.1,
    num_leaves=63,
    min_child_samples=10,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)
lgbm.fit(X_train_enc, y_train)
print("LightGBM trained (no class_weight — gradient boosting handles it natively)")

Removed 0 low-support classes: []
Kept 24 classes, 891,414 rows
LightGBM trained (no class_weight — gradient boosting handles it natively)


In [25]:
y_pred = lgbm.predict(X_test_enc)

acc = accuracy_score(y_test, y_pred)
bal_acc = balanced_accuracy_score(y_test, y_pred)
print(f"Overall Accuracy:          {acc:.4f}")
print(f"Overall Balanced Accuracy: {bal_acc:.4f}")
print(f"(Gap = {acc - bal_acc:.4f})")
print(f"\nC4.5  → Overall: 0.4886, Balanced: 0.2831")
print(f"RF    → Overall: 0.2880, Balanced: 0.3647")
print(f"LGBM  → Overall: {acc:.4f}, Balanced: {bal_acc:.4f}")
print(f"vs C4.5: Overall {acc-0.4886:+.4f}, Balanced {bal_acc-0.2831:+.4f}")

report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
rows = []
for cls, m in report.items():
    if cls in ("accuracy", "macro avg", "weighted avg"):
        continue
    rows.append({
        "Activity":  cls,
        "Support":   int(m["support"]),
        "Recall":    round(m["recall"], 3),
        "Precision": round(m["precision"], 3),
        "F1":        round(m["f1-score"], 3),
    })

df_report = pd.DataFrame(rows).sort_values("Support", ascending=False)
print(f"\nPer-class breakdown ({len(df_report)} classes):")
print(df_report.to_string(index=False))

weak = df_report[df_report["Recall"] < 0.3]
if not weak.empty:
    print(f"\nWeak classes (Recall < 0.30):")
    print(weak[["Activity", "Support", "Recall"]].to_string(index=False))
else:
    print("\nNo weak classes — all Recall >= 0.30!")

/Users/zeynepcetin/bppso-groupwork-1/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Overall Accuracy:          0.1761
Overall Balanced Accuracy: 0.0695
(Gap = 0.1065)

C4.5  → Overall: 0.4886, Balanced: 0.2831
RF    → Overall: 0.2880, Balanced: 0.3647
LGBM  → Overall: 0.1761, Balanced: 0.0695
vs C4.5: Overall -0.3125, Balanced -0.2136

Per-class breakdown (24 classes):
                Activity  Support  Recall  Precision    F1
     W_Call after offers    35925   0.076      0.188 0.108
  W_Complete application    29570   0.406      0.249 0.309
  W_Validate application    23413   0.457      0.196 0.274
 W_Call incomplete files    14373   0.004      0.025 0.006
          W_Handle leads     9442   0.201      0.219 0.209
          O_Create Offer     8185   0.220      0.065 0.100
               O_Created     8145   0.172      0.100 0.127
O_Sent (mail and online)     7597   0.032      0.131 0.052
               A_Concept     6300   0.042      0.069 0.052
              A_Accepted     6283   0.005      0.104 0.009
              A_Complete     6098   0.027      0.111 0.044
    

In [26]:
importances = lgbm.feature_importances_
total = importances.sum()
feat_imp = sorted(zip(features, importances / total), key=lambda x: x[1], reverse=True)
print("Feature Importances (LGBM):")
for feat, score in feat_imp:
    print(f"  {feat}: {score:.4f}")

Feature Importances (LGBM):
  prev_activity: 0.2112
  loan_goal: 0.1595
  amount_category: 0.1426
  duration_bin: 0.1261
  credit_score_bin: 0.1111
  case_age_category: 0.0910
  application_type: 0.0473
  is_repeated: 0.0454
  has_rejection: 0.0322
  offer_category: 0.0239
  has_accepted_offer: 0.0097


In [27]:
sim_package = {
    "model": lgbm,
    "encoder": encoder,
    "bins": {
        "duration_bin": duration_bins.tolist() if hasattr(duration_bins, 'tolist') else duration_bins,
        "case_age_category": case_age_bins.tolist() if hasattr(case_age_bins, 'tolist') else case_age_bins
    },
    "features": features,
    "classes": list(lgbm.classes_),
}

with open("simulation_brain_lgbm.pkl", "wb") as f:
    pickle.dump(sim_package, f)

print(f"LightGBM model saved to simulation_brain_lgbm.pkl")

LightGBM model saved to simulation_brain_lgbm.pkl
